# Day 18 — Mock Interview Practice

**Week 3 · Day 4 · Level: ADVANCED (16 ADVANCED + 2 HARD)**

18 questions across 8 categories. Week 3 stays at product-company level: multi-step problems, real-world scenarios, and system design aspects. Day 18 focuses on **market-basket analytics, two-heap stream algorithms, corrupt-data quarantine, robust anomaly detection, backpressure, reverse ETL, open table formats, data mesh, Snowflake Iceberg/DR, modern orchestrators, Delta forensics, and the semantic layer** — all fresh vs days 1-17.

| # | Category | Questions |
|---|----------|-----------|
| 1 | SQL Advanced | 2 |
| 2 | Python — Loops / Functions / Lists / Dicts | 2 |
| 3 | PySpark Coding | 2 |
| 4 | Spark Optimization & Deep Spark | 2 |
| 5 | ETL — Failures, Late Data, Schema, Fault Tolerance | 3 |
| 6 | Data Modeling | 2 |
| 7 | Snowflake | 2 |
| 8 | Monitoring, Logging & Orchestration | 3 |

**Rules:** attempt each question yourself first (say the approach out loud), then compare with the solution. Target ~25 minutes per question. Score yourself in the scorecard at the end.

## Q1 — SQL Advanced: Market-Basket Pair Analysis with Lift [ADVANCED]

**Interviewer (Amazon):** We have an `orders` table with one row per (order, product). For a recommendation feature, find the product pairs most frequently bought together — ranked by **lift**, not raw co-occurrence — so that universally-popular products don't dominate.

1. Define **support**, **confidence**, and **lift** for a pair (A, B).
2. Write ONE query returning each pair, its co-occurrence count, and its lift, ordered by co-occurrence then lift.
3. Why does the self-join need `a.product < b.product`? What appears if you drop it?
4. Follow-up: at 100M orders, why does this self-join explode, and what do real systems (FP-growth / MLlib) do instead?

**Expected for the sample data:** bread+eggs, eggs+milk, and bread+milk each co-occur in 3 orders — but their lifts differ (1.13 vs 0.9 vs 0.9) because lift rescales by popularity.

In [0]:
# Solution Q1 — Market-basket pairs with lift
# support(A,B) = P(A and B)          = pair_orders / total_orders
# lift(A,B)    = support(A,B) / (support(A) * support(B))
#   lift > 1 -> bought together MORE than independence predicts
#   lift = 0.9 (bread, milk) -> together LESS than random, despite 3 co-orders!

from pyspark.sql import functions as F

orders = spark.createDataFrame([
    (1, "bread"), (1, "milk"), (1, "eggs"),
    (2, "bread"), (2, "milk"),
    (3, "milk"), (3, "eggs"),
    (4, "bread"), (4, "eggs"), (4, "cheese"),
    (5, "milk"), (5, "cheese"),
    (6, "bread"), (6, "milk"), (6, "eggs"), (6, "cheese"),
], ["order_id", "product"])
orders.createOrReplaceTempView("orders")

result = spark.sql("""
WITH pair_counts AS (
  SELECT a.product AS product_a, b.product AS product_b,
         COUNT(DISTINCT a.order_id) AS pair_orders
  FROM orders a
  JOIN orders b
    ON a.order_id = b.order_id
   AND a.product < b.product          -- each unordered pair ONCE; no self-pairs
  GROUP BY a.product, b.product
),
product_counts AS (
  SELECT product, COUNT(DISTINCT order_id) AS orders_with
  FROM orders GROUP BY product
),
total AS (SELECT COUNT(DISTINCT order_id) AS n FROM orders)
SELECT p.product_a, p.product_b, p.pair_orders,
       ROUND((p.pair_orders / t.n) /
             ((pa.orders_with / t.n) * (pb.orders_with / t.n)), 2) AS lift
FROM pair_counts p
JOIN product_counts pa ON pa.product = p.product_a
JOIN product_counts pb ON pb.product = p.product_b
CROSS JOIN total t
ORDER BY pair_orders DESC, lift DESC
""")
result.show()

pairs = {(r["product_a"], r["product_b"]): (r["pair_orders"], r["lift"]) for r in result.collect()}
assert pairs[("bread", "eggs")] == (3, 1.13)   # best lift at 3 co-orders
assert pairs[("bread", "milk")] == (3, 0.9)    # popular pair, but lift < 1!
assert pairs[("cheese", "milk")] == (2, 0.8)

# Interview points:
# - Without a.product < b.product: every pair appears twice ((A,B) and (B,A))
#   AND each product pairs with itself -> double counting + garbage self-pairs.
# - Lift rescues you from popularity bias: milk is in 5/6 orders, so co-occurring
#   with milk 3/6 times is actually WORSE than random (lift 0.9).
# - Scale problem: pairs per order grow O(k^2) per basket, and the self-join shuffles
#   the whole table. Real recommenders use FP-growth (Spark MLlib FPGrowth) or
#   approximate candidates (SON/PCY, sampled mining), then join back for exact counts.
print("Q1 OK — bread+eggs lift 1.13 beats bread+milk 0.9 despite equal co-counts")

## Q2 — SQL Advanced: GROUPING SETS Subtotal Report [ADVANCED]

**Interviewer (Uber):** Finance wants one report with sales by region+country, by region alone, AND the grand total — from ONE pass over the table.

1. Write a Databricks SQL query using `GROUPING SETS` that produces all three levels in a single result set, with clean labels like `ALL COUNTRIES`.
2. What does `GROUPING()` return, and why can't you just test `country IS NULL` to detect subtotal rows?
3. How do `GROUPING SETS`, `ROLLUP`, and `CUBE` differ?
4. Why is one GROUPING SETS query cheaper than a `UNION ALL` of three aggregate queries?

**Expected for the sample data:** 4 country rows (level 0), 2 region subtotals (level 1: APAC 6000, EMEA 2900), and grand total 8900 (level 2).

In [0]:
# Solution Q2 — GROUPING SETS subtotals in one scan
from pyspark.sql import functions as F

sales = spark.createDataFrame([
    ("APAC", "India",  "online", 1200.0), ("APAC", "India",  "retail",  800.0),
    ("APAC", "Japan",  "online", 2500.0), ("APAC", "Japan",  "retail", 1500.0),
    ("EMEA", "France", "online",  900.0), ("EMEA", "France", "retail", 1400.0),
    ("EMEA", "Spain",  "online",  600.0),
], ["region", "country", "channel", "amount"])
sales.createOrReplaceTempView("sales")

report = spark.sql("""
SELECT
  CASE WHEN GROUPING(region)  = 1 THEN 'ALL REGIONS'   ELSE region  END AS region,
  CASE WHEN GROUPING(country) = 1 THEN 'ALL COUNTRIES' ELSE country END AS country,
  ROUND(SUM(amount), 1) AS total_sales,
  GROUPING(region) + GROUPING(country) AS subtotal_level   -- 0=detail 1=region 2=grand
FROM sales
GROUP BY GROUPING SETS ((region, country), (region), ())
ORDER BY subtotal_level, region, country
""")
report.show()

rows = {(r["region"], r["country"], r["total_sales"], r["subtotal_level"]) for r in report.collect()}
assert ("ALL REGIONS", "ALL COUNTRIES", 8900.0, 2) in rows   # grand total
assert ("APAC", "ALL COUNTRIES", 6000.0, 1) in rows          # region subtotal
assert ("EMEA", "France", 2300.0, 0) in rows                  # detail row

# Interview points:
# - GROUPING(col) = 1 means col was NOT part of the grouping set for that row —
#   the safest way to label subtotals. Testing `country IS NULL` is WRONG because
#   a real NULL country in the data would masquerade as a subtotal row.
# - GROUPING SETS: you list exactly the aggregation levels (explicit).
#   ROLLUP(a, b): hierarchy (a,b), (a), () — the most common finance case.
#   CUBE(a, b): ALL combos — (a,b), (a), (b), () — for exploration, explodes 2^n.
# - UNION ALL of three queries = 3 scans + 3 shuffles of the same data;
#   GROUPING SETS = one scan, one shuffle, Spark splits/extends each group
#   to all applicable grouping levels internally.
print("Q2 OK — 7 rows: 4 detail + 2 region subtotals + 1 grand total, single pass")

## Q3 — Python: Median from a Data Stream [ADVANCED]

**Interviewer (Google):** Numbers arrive one at a time. After each arrival, you must be able to return the median of everything seen so far.

1. Design a `MedianFinder` class with `add(num)` and `median()`, both **O(log n)**.
2. State the two-heap invariant and the rebalance rules precisely.
3. Why a heap for each half, and why must the max-heap be implemented with negated values in Python?
4. Follow-up: how would you support a **sliding-window** median, or an approximate median (Greenwald-Khanna / P^2) at scale?

**Expected:** for the stream 5, 15, 1, 3, 2, 8 the medians are 5, 10, 5, 4, 3, 4.

In [0]:
# Solution Q3 — running median with two heaps
import heapq

class MedianFinder:
    def __init__(self):
        self.low = []   # MAX-heap of the smaller half (negated values)
        self.high = []  # MIN-heap of the larger half

    def add(self, num):
        # 1) push into the smaller half, then hand its max to the larger half
        heapq.heappush(self.low, -num)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        # 2) rebalance so len(low) == len(high) or exactly one more
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):        # odd count -> top of low
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2  # even -> average of tops

mf = MedianFinder()
observed = []
for x in [5, 15, 1, 3, 2, 8]:
    mf.add(x)
    observed.append(mf.median())
print("running medians:", observed)
assert observed == [5.0, 10.0, 5.0, 4.0, 3.0, 4.0]

# Interview points:
# - Invariant: every element in low <= every element in high;
#   |len(low) - len(high)| <= 1, low holds the extra when odd.
# - The median is therefore ALWAYS at the two heap tops — O(1) read.
# - heapq is a min-heap only, so the max-heap stores NEGATED values
#   (-self.low[0] is the max of the smaller half).
# - Complexity: add is O(log n) (constant number of heap ops), median O(1).
# - Sliding window median: heaps can't delete arbitrary elements cheaply ->
#   use an ordered multiset (SortedList) or heaps + lazy deletion with a
#   delayed-removal counter (the LeetCode 480 trick).
# - Approximate median at scale: P^2 or Greenwald-Khanna quantile sketches,
#   or Spark's approxQuantile / t-digest which does exactly this per partition.
print("Q3 OK — two-heap invariant holds, all medians correct")

## Q4 — Python: Course Schedule — Topological Sort [ADVANCED]

**Interviewer (Meta):** Given `num_courses` and a list of `[course, prerequisite]` pairs: (a) can all courses be completed, and (b) return one valid order.

1. Why is this a graph problem, and what does a cycle in the graph mean for the answer?
2. Implement **Kahn's algorithm** (BFS topological sort) to return a valid order, or `None` on a cycle.
3. State the time and space complexity.
4. Follow-up: how would the DFS 3-coloring alternative work, and what are the tradeoffs (recursion depth, detecting vs reporting the cycle)?

**Expected:** 4 courses with prereqs [[1,0],[2,0],[3,1],[3,2]] → a valid order like [0,1,2,3]; [[1,0],[0,1]] → impossible (cycle).

In [0]:
# Solution Q4 — Kahn's algorithm (BFS topological sort)
from collections import deque, defaultdict

def find_order(num_courses, prereqs):
    """Return a valid course order, or None if a cycle makes it impossible."""
    adj = defaultdict(list)          # prereq -> courses that depend on it
    indegree = [0] * num_courses     # number of unmet prereqs per course
    for course, pre in prereqs:
        adj[pre].append(course)
        indegree[course] += 1

    # start with every course that has no prerequisites
    queue = deque(i for i in range(num_courses) if indegree[i] == 0)
    order = []
    while queue:
        c = queue.popleft()
        order.append(c)
        for nxt in adj[c]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:    # last prereq satisfied -> ready
                queue.append(nxt)

    # if a cycle exists, its nodes never reach indegree 0 -> order is short
    return order if len(order) == num_courses else None

# Test 1: valid DAG
order = find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
print("valid order:", order)
assert order is not None and set(order) == {0, 1, 2, 3}
assert order.index(0) < order.index(1) and order.index(0) < order.index(2)
assert order.index(1) < order.index(3) and order.index(2) < order.index(3)

# Test 2: circular dependency -> impossible
assert find_order(2, [[1, 0], [0, 1]]) is None

# Interview points:
# - Model: nodes = courses, edge pre -> course. Valid curriculum = topological order;
#   a cycle means no valid ordering exists.
# - Complexity: O(V + E) time, O(V + E) space — every node and edge touched once.
# - Kahn's is BFS-flavored: indegree 0 = 'no unmet dependency', queue = ready set.
# - DFS 3-color (white/grey/black) detects cycles via grey back-edges and gives the
#   cycle nodes, but recursion can blow the stack on deep graphs — Kahn's is iterative
#   and safer. DFS reversed postorder is the classic alternative topological sort.
# - Real-world analog: build systems (make/Bazel), Airflow task scheduling, dbt model
#   ordering — all are topological sorts over dependency graphs.
print("Q4 OK — DAG ordered, cycle detected")

## Q5 — PySpark Coding: Corrupt-Record Quarantine on Ingest [ADVANCED]

**Interviewer (Netflix):** Events land as JSON files; a few rows are malformed or have wrong types. The batch must **NOT** fail — valid rows proceed, bad rows go to a quarantine table **with the original payload** for triage.

1. Explain `PERMISSIVE` vs `FAILFAST` vs `DROPMALFORMED` parse modes — when would you pick each?
2. Implement: read with an explicit schema, split valid vs corrupt rows via `columnNameOfCorruptRecord`, and show the quarantined payloads.
3. In the demo, BOTH a type-mismatch row (`"user_id": "oops"`) and truly broken JSON (`this is not json`) get quarantined — explain why a wrong type poisons the whole record in the JSON reader.
4. Production follow-up: how do you alert on quarantine **rate** (not count), and why can a *suddenly rising* corrupt rate signal an unannounced schema-contract change rather than garbage data?

**Expected:** 3 valid rows (user_ids 1,2,3) + 2 quarantined rows carrying the original text.

In [0]:
# Solution Q5 — PERMISSIVE mode + _corrupt_record quarantine
from pyspark.sql import functions as F, types as T

json_lines = [
    '{"user_id": 1, "event": "login",    "ts": "2026-10-30 09:00:00"}',
    '{"user_id": 2, "event": "purchase", "ts": "2026-10-30 09:05:00"}',
    '{"user_id": "oops", "event": null,  "ts": "not-a-date"}',     # wrong types
    'this is not json at all',                                   # broken JSON
    '{"user_id": 3, "event": "login",    "ts": "2026-10-30 09:10:00"}',
]
base = "/Volumes/workspace/default/etl_test_volume/d18_q5"
dbutils.fs.rm(base, True)
dbutils.fs.mkdirs(base)
dbutils.fs.put(base + "/events.json", "\n".join(json_lines), True)

schema = T.StructType([
    T.StructField("user_id", T.LongType()),
    T.StructField("event", T.StringType()),
    T.StructField("ts", T.TimestampType()),
    # must be IN the schema for columnNameOfCorruptRecord to work
    T.StructField("_corrupt_record", T.StringType()),
])

raw = (spark.read.schema(schema)
       .option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt_record")
       .json(base + "/events.json"))

valid = raw.filter(F.col("_corrupt_record").isNull()).select("user_id", "event", "ts")
dlq   = raw.filter(F.col("_corrupt_record").isNotNull()).select("user_id", "event", "ts", "_corrupt_record")

valid.show(truncate=False)
dlq.show(truncate=False)

assert [r.user_id for r in valid.orderBy("user_id").collect()] == [1, 2, 3]
assert dlq.count() == 2

# In production:
#   (valid).write.mode("append").saveAsTable("events")
#   (dlq)  .write.mode("append").saveAsTable("events_dlq")

# Interview points:
# - PERMISSIVE: null out bad fields, capture the raw text -> the only mode for
#   quarantine. FAILFAST: abort the batch (good for contracts you control).
#   DROPMALFORMED: silently loses data — almost never acceptable in pipelines.
# - Wrong TYPE ("oops" for Long) also flags the record: the JSON reader is
#   record-corrupt, not field-null — one unparseable field poisons the row.
#   (Contrast: a *missing* field is just null, NOT corrupt.)
# - Alert on quarantine RATE (corrupt / total per batch) with a relative threshold;
#   if it jumps from 0.1% to 30% overnight, suspect a producer schema change first,
#   data garbage second — check the offending payloads for a consistent new field.
print("Q5 OK — 3 valid + 2 quarantined with original payloads")

## Q6 — PySpark Coding: Statistical Anomaly Detection at Scale [ADVANCED]

**Interviewer (Adobe):** Flag anomalous IoT sensor readings across millions of rows **without training an ML model** — it must run as plain Spark code every hour.

1. Why does a plain z-score (mean/stddev) break on dirty data, and why is **median + MAD** robust? What fraction of outliers can each tolerate before it breaks?
2. Implement with `approxQuantile`: robust z = 0.6745 * (x − median) / MAD, flag |robust z| > 3.5.
3. What does the `relativeError=0.001` argument buy you? When would exact percentiles matter?
4. Follow-up: per-machine baselines — which Spark primitive computes medians per group at scale?

**Expected:** 30 normal readings around 72°F plus injected 95.0, 40.0, 88.0 — exactly those 3 flagged, with robust z-scores far outside ±3.5.

In [0]:
# Solution Q6 — median + MAD robust anomaly scoring
from pyspark.sql import functions as F
import random

random.seed(42)
readings = [round(random.gauss(72, 2), 1) for _ in range(30)] + [95.0, 40.0, 88.0]
temps = spark.createDataFrame([(r,) for r in readings], ["temp_f"])

# Step 1: robust center (median) and spread (MAD = median absolute deviation)
median = temps.approxQuantile("temp_f", [0.5], 0.001)[0]
devs = temps.withColumn("abs_dev", F.abs(F.col("temp_f") - F.lit(median)))
mad = devs.approxQuantile("abs_dev", [0.5], 0.001)[0]
assert mad > 0
print(f"median = {median}, MAD = {mad}")

# Step 2: robust z-score. 0.6745 = Phi^-1(0.75): makes MAD comparable to stddev
# under normality, so the classic 3-sigma intuition maps to |robust_z| > 3.5.
scored = (devs
    .withColumn("robust_z", F.round(0.6745 * (F.col("temp_f") - F.lit(median)) / F.lit(mad), 2))
    .withColumn("is_anomaly", ~F.col("robust_z").between(-3.5, 3.5)))

anoms = scored.filter("is_anomaly").orderBy("temp_f").select("temp_f", "robust_z")
anoms.show()
assert anoms.count() == 3
assert {r.temp_f for r in anoms.collect()} == {95.0, 40.0, 88.0}

# Interview points:
# - mean/stddev have a breakdown point of 0%: ONE extreme value shifts both,
#   masking itself and everything else. Median/MAD break down only after 50%
#   of the data is corrupted — that is what 'robust' means mathematically.
# - approxQuantile runs a GK/t-digest-style sketch per partition and merges —
#   relativeError 0.001 trades a tiny quantile error for O(1) memory per task,
#   avoiding a full collect/sort. Exact percentile_exact matters when the
#   threshold itself is contractual (billing tiers, SLA boundaries).
# - Per-group baselines: GROUP BY machine + percentile_approx (SQL) or
#   applyInPandas with pandas.Series.median()/mad() per group.
# - This IS production anomaly detection for the 'no labels, no model' case;
#   graduated to ML only when you have labels or seasonality to learn.
print("Q6 OK — only the 3 injected outliers flagged (robust_z 13.3 to 27.2)")

## Q7 — Spark Optimization: Dynamic Resource Allocation & Executor Sizing [ADVANCED]

**Interviewer (Databricks):** Your job's stages alternate between tiny transforms and huge shuffles. Fixed clusters either waste money or stall. Explain **Dynamic Resource Allocation (DRA)** and how to size executors.

Cover:
1. What triggers scale-UP and scale-DOWN in DRA? Name the key configs and what each knob does.
2. Why does DRA require the **External Shuffle Service** to release executors safely? What is the Databricks-native equivalent (cluster autoscaling, autoscale local storage)?
3. When DRA actively **hurts**: cached DataFrames, streaming (stateful), and re-launched executors losing shuffle/broadcast state.
4. Executor sizing rules of thumb: cores-per-executor, memory-per-core, and the fewer-big vs many-small tradeoff.
5. Deliver the 90-second interview answer.

In [0]:
# Solution Q7 — DRA configs + 90-second talk track

configs = {
    "spark.dynamicAllocation.enabled":                    "true",
    "spark.dynamicAllocation.minExecutors":              "2",     # warm floor: avoid cold starts
    "spark.dynamicAllocation.maxExecutors":              "50",    # hard cost ceiling
    "spark.dynamicAllocation.executorAllocationRatio":   "0.5",   # request half a executor per pending task -> smoother ramp
    "spark.dynamicAllocation.schedulerBacklogTimeout":   "60s",   # pending tasks for this long -> ask for more executors
    "spark.dynamicAllocation.sustainedSchedulerBacklogTimeout": "60s",  # keep asking while backlog persists
    "spark.dynamicAllocation.executorIdleTimeout":       "120s",  # idle this long -> release
    "spark.dynamicAllocation.cachedExecutorIdleTimeout": "30m",   # executors holding cached blocks linger longer
    "spark.shuffle.service.enabled":                     "true",  # ESS: lets executors be removed without killing shuffle files
}
for k, v in configs.items():
    print(f"{k} = {v}")

TALK_TRACK = """
[90-SECOND ANSWER]
DRA right-sizes executors to actual demand. Scale-UP: when the scheduler has a
backlog of pending tasks for schedulerBacklogTimeout, the allocator requests
more executors (executorAllocationRatio damps over-requesting). Scale-DOWN:
executors idle longer than executorIdleTimeout are released.

The catch: an executor owns its shuffle output. Killing it would break reducers
reading those files — so removal requires the External Shuffle Service, a
standalone daemon that serves shuffle files after their executor dies. On
Databricks, cluster autoscaling handles this natively (and autoscale-local-storage
sizes ephemeral disks for shuffle-heavy jobs).

When DRA hurts: (1) caches — executors holding cached partitions only drop after
cachedExecutorIdleTimeout, so volatile cache + aggressive DRA = thrash; (2) stateful
streaming — executors hold state stores, churn breaks locality; (3) jobs that
re-request executors every stage pay launch latency (JVM + warm JIT) repeatedly.

Sizing rules of thumb: 4-5 cores per executor (YARN overhead per JVM), and start
from a memory-per-core budget of 4-8 GB. Fewer-big executors amortize JVM/overhead
and help broadcast joins; many-small ones reduce straggler blast radius and GC
pause impact. Big no: > ~60GB heap in one JVM — G1 or not, GC pauses will hurt.
"""
print(TALK_TRACK)

# Self-check: can you answer these without looking?
# - Why is minExecutors > 0 recommended even when idle? (cold-start latency)
# - What is the ONE config that makes removing executors safe? (ESS / shuffle service)
# - Why does caching interact badly with DRA? (cached blocks pin executors)
print("Q7 OK — talk track + config set")

## Q8 — Spark Optimization: File Statistics, Data Skipping & Compression [ADVANCED]

**Interviewer (Amazon):** A filtered query on a 10 TB Delta table reads only 2 GB. Explain **exactly** why — down to the file format internals.

Cover:
1. Columnar internals: Parquet row groups, column chunks, footer min/max statistics, and predicate pushdown at each level.
2. Delta Lake file-level statistics (min/max/null counts per column per file) — how **file skipping** works before any data is read, and why **data layout** (ORDER BY / ZORDER / liquid clustering) decides how good those statistics are.
3. The small-file problem: why skipping degrades when many files' ranges overlap, and how OPTIMIZE / liquid clustering fix it.
4. Compression: snappy vs zstd vs gzip — CPU vs ratio vs splittability, and your pick for a scan-heavy lakehouse.
5. Connect it end-to-end: query → planner → file stats → row-group pruning → IO saved.

In [0]:
# Solution Q8 — talk track: how a filter turns into 2 GB of IO
TALK_TRACK = """
[THE ANSWER, LAYER BY LAYER]

1) COLUMNAR PRUNING (Parquet):
   Files are split into row groups (~128MB logical); each column is stored per
   row group as a column chunk with footer statistics: min, max, null count.
   A query needs 3 of 100 columns? Parquet reads only 3 column chunks — IO is
   proportional to the SELECTED columns and to SKIPPED row groups, not rows.

2) DELTA FILE SKIPPING (the 10TB -> 2GB step):
   Every Delta data file has a transaction-log entry with per-column min/max
   stats. 'WHERE event_date = 2026-10-30' first prunes FILES: any file whose
   [min, max] range can't contain the date is never opened at all. This is why
   layout is destiny — stats quality is decided at WRITE time, not read time.

3) LAYOUT = QUALITY OF STATS:
   - ORDER BY(event_date): perfect pruning for that one column.
   - ZORDER BY(event_date, customer_id): clusters on several dimensions
     simultaneously (space-filling curve) so ranges stay narrow for all of them.
   - Liquid clustering (CLUSTER BY): Databricks' newer incremental layout —
     no partition columns needed, reclusters on the fly as data grows.
   Small files hurt twice: overlapping ranges defeat pruning, and per-file
   open/list cost dominates. OPTIMIZE compacts files and (re)applies layout.

4) COMPRESSION — the tradeoff table:
   snappy: fastest decompress (~ saturates NVMe), mediocre ratio -> DEFAULT for hot data.
   zstd:    ~3-5x stronger ratio than snappy at modest CPU cost -> cold / archived
            tiers and cloud egress-sensitive setups.
   gzip:    high ratio but NOT splittable in raw form + slow -> avoid for wide analytics.
   The punchline: Parquet is splittable regardless of codec (codec applies INSIDE
   column pages), so 'gzip is unsplittable' is a raw-text problem, not Parquet.
   For a scan-heavy lakehouse I pick zstd by default (Databricks' default) —
   scan CPU is cheap, storage+network is the scarce resource.

5) END-TO-END CHAIN TO SAY IN THE INTERVIEW:
   SQL filter -> Delta log file stats -> files skipped -> Parquet footer row-group
   stats -> row groups skipped -> column pruning -> dictionary/page-level decoding
   -> only surviving pages decompressed. 10 TB table, 2 GB read: that is the product
   of all five pruning layers, not magic.
"""
print(TALK_TRACK)

# Self-check questions:
# - Why can't ZORDER be as perfect as ORDER BY for one column? (multi-dim tradeoff)
# - What breaks file skipping: overlapping files? small files? both?
# - Why is zstd now the Databricks default over snappy?
print("Q8 OK — five-layer pruning chain")

## Q9 — ETL: Backpressure in Streaming Pipelines [ADVANCED]

**Interviewer (Uber):** Kafka input rate spikes 10×; your Structured Streaming job's processing rate can't keep up. Explain backpressure and walk through what you do, in order.

Cover:
1. What backpressure **is**, and where the pressure physically accumulates (Kafka offsets / consumer lag, shuffle buffers, state stores, checkpoint) — plus the failure modes (OOM, lag explosion, checkpoint falling behind).
2. Source-side throttles in Structured Streaming: `maxOffsetsPerTrigger` (Kafka), `maxFilesPerTrigger` / `maxBytesPerTrigger` (Auto Loader), and trigger interval interplay.
3. Legacy DStreams had `spark.streaming.backpressure.enabled` auto-throttling — what is the Structured Streaming equivalent (rate-aware sources, `AvailableNow` catch-up mode)?
4. Consumer-lag monitoring: which metric, which thresholds, which alert?
5. The remedies ladder: throttle → scale the consumer → isolate the slow stage → compact/repartition the topic → shed load to DLQ → degrade to batch catch-up (AvailableNow).

The runnable demo below simulates lag growth across batches and applies the decision rules.

In [0]:
# Solution Q9 — lag simulation + backpressure decision engine + talk track

# (a) Source-side throttle configs to state in the interview:
throttle_configs = {
    "kafka":     '.option("maxOffsetsPerTrigger", 500000)',           # cap batch size per trigger
    "autoloader": '.option("maxBytesPerTrigger", "8g").option("maxFilesPerTrigger", 10000)',
    "trigger":   '.trigger(processingTime="30 seconds")',              # fixed cadence paces batches
    "catchup":   '.trigger(availableNow=True)',                       # batch-style catch-up, then stop
}
for k, v in throttle_configs.items():
    print(f"{k:>10}: {v}")

# (b) Simulate batches: events/sec in vs events/sec actually processed
batches = [
    # minute, input_rate, process_rate
    (0,   20_000, 19_500),   # healthy headroom
    (1,   60_000, 21_000),   # spike starts, lag begins
    (2,  120_000, 22_000),   # 10x spike
    (3,  140_000, 22_500),
    (4,  150_000, 23_000),   # consumer saturating
]

lag = 0
SLOWING = 60          # sec of lag where you page a human
DEGRADE = 1_800       # sec of lag where you stop trying to keep up
decisions = []
for minute, in_r, pr_r in batches:
    lag = max(0, lag + (in_r - pr_r) * 60)      # unprocessed events -> seconds of work backlog
    if lag >= DEGRADE:
        action = "DEGRADE: switch trigger to availableNow catch-up + route overflow to DLQ"
    elif lag >= SLOWING:
        action = "SCALE: raise maxOffsetsPerTrigger budget + add consumer parallelism (maxOffsetsPerTrigger ~ partitions)"
    elif in_r > pr_r:
        action = "THROTTLE: cap maxOffsetsPerTrigger so each batch completes inside the trigger interval"
    else:
        action = "OK: keep fixed 30s trigger"
    decisions.append((minute, f"{in_r:>7,}", f"{pr_r:>7,}", f"{lag:>9,}", action))

print(f"{'min':>3} {'in_eps':>8} {'proc_eps':>8} {'lag_sec':>10}  decision")
for row in decisions:
    print(f"{row[0]:>3} {row[1]:>8} {row[2]:>8} {row[3]:>10}  {row[4]}")

assert "THROTTLE" in decisions[1][4]
assert "SCALE" in decisions[2][4] and "SCALE" in decisions[3][4]
assert "DEGRADE" in decisions[4][4]

# (c) Talk track
# - Backpressure = the pipeline refuses to ingest faster than it can process, and the
#   surplus ACCUMULATES at the source (Kafka retains it; consumers fall behind = LAG).
# - Monitored via consumer group lag (kafka offset diff), alert paged at lag covering
#   > N minutes of processing; death spiral check: is lag growing or shrinking per batch?
# - Structured Streaming has NO auto-backpressure like DStreams — YOU set the pace with
#   maxOffsetsPerTrigger/maxBytesPerTrigger + trigger interval, so the two must be tuned
#   as a pair (a batch that overruns its trigger interval is the bug).
# - Structured Streaming kills the job if a single batch runs > epoch-long timeouts, so
#   a runaway batch fails loudly rather than queueing silently forever — catch it, then
#   drain with AvailableNow and quarantine the poison window.
print("Q9 OK — throttle -> scale -> degrade ladder simulated")

## Q10 — ETL: Reverse ETL — Hash-Diff CDC Extract [ADVANCED]

**Interviewer (Airbnb):** Push customer profiles from Delta Lake to a marketing platform every 10 minutes. The API accepts **updates only**, not full syncs — extract only what changed since the last export.

1. What is reverse ETL, and how does it differ from ordinary batch export and from CDC-into-warehouse?
2. Implement a **hash-diff extract**: md5 over the tracked columns, LEFT JOIN vs the last export snapshot → produce INSERT / UPDATE / DELETE feeds. Why hash instead of comparing every column with ORs?
3. Idempotency: the export job succeeded but the API call failed halfway. How does an **export-checkpoint state table** make the job safely re-runnable?
4. PII: what must be masked or hashed before pushing to a third-party system?

In [0]:
# Solution Q10 — hash-diff change feed for reverse ETL
from pyspark.sql import functions as F

# 'current' = what the lake holds now; 'prev' = what we last pushed to the platform
curr = spark.createDataFrame([
    (1, "alice", "alice@x.com", "PRO"), (2, "bob", "bob@x.com", "FREE"),
    (3, "carol", "carol@x.com", "PRO"), (5, "eve", "eve@x.com", "FREE"),
], ["id", "name", "email", "plan"])
prev = spark.createDataFrame([
    (1, "alice", "alice@x.com", "FREE"), (2, "bob", "bob@x.com", "FREE"),
    (3, "carol", "old@x.com", "PRO"), (4, "dan", "dan@x.com", "FREE"),
], ["id", "name", "email", "plan"])

def hash_diff(df):
    tracked = ["name", "email", "plan"]          # columns that 'matter' downstream
    return (df.withColumn("row_hash", F.md5(F.concat_ws("||", *tracked)))
              .select("id", "row_hash", *tracked))

c, p = hash_diff(curr), hash_diff(prev)
c.createOrReplaceTempView("curr_hashed")
p.createOrReplaceTempView("prev_hashed")

# INSERT = in current, never exported; UPDATE = hash changed; DELETE = gone from source
cdc = spark.sql("""
SELECT c.id, c.name, c.email, c.plan,
       CASE WHEN p.id IS NULL THEN 'INSERT' ELSE 'UPDATE' END AS op
FROM curr_hashed c LEFT JOIN prev_hashed p ON c.id = p.id
WHERE p.id IS NULL OR c.row_hash <> p.row_hash
UNION ALL
SELECT p.id, NULL AS name, NULL AS email, NULL AS plan, 'DELETE' AS op
FROM prev_hashed p LEFT JOIN curr_hashed c ON c.id = p.id
WHERE c.id IS NULL
""").orderBy("id")
cdc.show()
ops = {r["id"]: r["op"] for r in cdc.collect()}
assert ops == {1: "UPDATE", 3: "UPDATE", 4: "DELETE", 5: "INSERT"}, ops

# Interview points:
# - Reverse ETL = lake -> OPERATIONAL system (Salesforce, Braze, ad platforms);
#   the hard parts are incremental extraction + API-side idempotency, not SQL.
# - Hash-diff beats per-column OR-chains: 1 comparison, no NULL-safe traps,
#   and the hash column doubles as a watermark. Caveat: md5 over concat is only
#   correct with a separator (concat_ws '||') so 'ab|c' != 'a|bc', and you must
#   FIX the column list — new columns change all hashes and cause a full resync.
# - Re-runnability: log every export batch (run_id, export_hash_snapshot, status)
#   in an etl_export_state table. On failure: re-read the SAME prev snapshot from
#   the last successful batch, re-push, mark complete. No drift, no double-push —
#   the API sees the same ops replayed (idempotent by upsert key).
# - PII: emails/names -> SHA-256 pseudonyms or field-level masking in the export
#   view BEFORE it leaves the lake; log consent flags and enforce DSR deletes on
#   the DELETE feed FIRST (compliance outranks the marketing batch).
print("Q10 OK — 1 update, 1 change-feed UPDATE, 1 DELETE, 1 INSERT")

## Q11 — ETL: Delta vs Iceberg vs Hudi — Open Table Format Decision [HARD]

**Interviewer (Netflix):** You're choosing the lakehouse table format for a multi-engine platform — Spark, Trino, Flink, and Snowflake all need to read (some write) the same tables. Compare the three open table formats and decide.

Cover:
1. **Row-level ops**: how MERGE / upsert / delete is implemented in each — copy-on-write vs merge-on-read, position vs equality deletes, and the read-amplification cost each choice creates.
2. **Catalog story** (the 2025 reality): Hive/Glue metastore vs Nessie/Polaris/Unity Catalog vs the **REST catalog** standard — why catalog integration now matters more than file layout.
3. Feature deep-dive: hidden partitioning / partition evolution, schema evolution, time travel, branching/tagging support.
4. **Operational burden**: compaction, snapshot expiry, file-layout maintenance in each (Delta OPTIMIZE, Iceberg rewrite_data_files + expire_snapshots, Hudi clustering/table services).
5. Your decision framework for: (a) all-Spark Databricks shop, (b) multi-engine open org, (c) streaming-heavy CDC workloads.

In [0]:
# Solution Q11 — comparison matrix + decision framework (talk track)

matrix = [
    ("Row-level writes", "MERGE into copy-on-write files", "copy-on-write AND merge-on-read (position deletes)", "CoW and MoR natively (log-based design)"),
    ("CDC / upsert speed", "good; batch MERGE", "strong; MoR splits write vs read path", "strongest; built for record-level CDC"),
    ("Read amplification", "read optimized files after rewrite", "MoR: readers compact deletes at read time", "MoR: same, needs async compaction (CLEAN)"),
    ("Catalog", "Unity Catalog / Hive (REST catalog support growing)", "Glue/Hive/Nessie/Polaris/REST — broadest", "Hive/Glue; no vendor-neutral REST standard"),
    ("Hidden partitioning", "no (use liquid clustering / ZORDER)", "yes — transforms in metadata, partition evolution", "no native; clustering keys instead"),
    ("Schema evolution", "strong (mergeSchema, column mapping)", "strongest (independent evolve + reorder)", "strong (with reconciliation APIs)"),
    ("Time travel", "yes (VERSION/ TIMESTAMP AS OF)", "yes (snapshot tables, expire_snapshots)", "yes (query as-of instant)"),
    ("Branching / tags", "yes (Unity Catalog-managed)", "yes + Nessie gives cross-table commits", "no"),
    ("Multi-engine writes", "DeltaUni readers (OSS), strong on Databricks", "best: open spec adopted by Spark/Trino/Flink/Snowflake", "good, but Spark-centric momentum"),
    ("Ops burden", "OPTIMIZE + VACUUM (liquid clustering reduces both)", "compaction, expire snapshots, rewrite files", "compaction/CLUSTERING/CLEAN table services"),
]
widths = [20, 34, 40, 40]
print(f"{'Aspect':<20} {'Delta Lake':<34} {'Iceberg':<40} {'Hudi':<40}")
print("-" * 134)
for row in matrix:
    print(" | ".join(str(v)[:w].ljust(w) for v, w in zip(row, widths)))

DECISION = """
DECISION FRAMEWORK (say it like this):
(a) All-Spark / Databricks shop -> Delta. Native MERGE + Unity Catalog + liquid
    clustering + Auto Loader; zero reason to add second-format ops burden.
(b) Multi-engine open org (Trino + Flink + Snowflake + Spark reading the same files)
    -> Iceberg. The open spec + REST catalog (Polaris/Nessie/Glue) is the defacto
    2025-26 interop standard; hidden partitioning also frees analysts from
    partition-key mistakes.
(c) Streaming-heavy / record-level CDC at high volume (change events per second,
    sub-minute freshness) -> Hudi MoR: writes append small delta files, readers
    see the merged view, async compaction keeps reads fast. Iceberg MoR is
    closing this gap, but Hudi's table services (CLEAN/CLUSTER/COMPACT) are the
    most battle-tested.
Lock-in lens: all three are open (DeltaUni, Iceberg OSS, Hudi OSS) — the real
lock-in is the CATALOG and the governance plane, so choose THAT first in 2026.
"""
print(DECISION)
print("Q11 OK — matrix + three-scenario recommendation")

## Q12 — Data Modeling: Grain Declaration & Degenerate Dimensions [ADVANCED]

**Interviewer (Stripe):** Model a fact table for **order-line events** — each order line can be paid, refunded, and shipped (several events per line over its life).

1. Why must you declare the **grain** before anything else? What is the grain here, and how does it differ from a fact-per-order model?
2. What are **degenerate dimensions** (e.g., `order_id`, `invoice_number`)? Why do they live in the fact with **no dimension table**, and why must they be in the natural key?
3. Write a **grain test** that fails the build if the natural key ever duplicates.
4. How does a wrong grain silently corrupt every downstream aggregate? Give a concrete example.

In [0]:
# Solution Q12 — event-grain fact with degenerate dimensions + grain test
from pyspark.sql import functions as F

# One row per (order line, event_type) — the event grain.
# Degenerate dimensions: order_id, invoice_no — real business keys that exist
# ONLY here (no dimension table) because the fact IS their only occurrence.
order_line_events = spark.createDataFrame([
    ("ORD-5001", 1, "INV-9001", "2026-10-30", 1299.0, "PAYMENT"),
    ("ORD-5001", 2, "INV-9001", "2026-10-30",  199.0, "PAYMENT"),
    ("ORD-5002", 1, "INV-9002", "2026-10-30",   89.0, "PAYMENT"),
    ("ORD-5003", 1, "INV-9003", "2026-10-30", 2499.0, "REFUND"),
    ("ORD-5003", 1, "INV-9003", "2026-10-30", 2499.0, "SHIP"),   # 2nd event, same line
], ["order_id", "line_num", "invoice_no", "event_date", "line_amount", "event_type"])

fact_ddl = """
CREATE TABLE fact_order_line_event (
  order_id      STRING,          -- degenerate dimension (business key)
  line_num      INT,
  invoice_no    STRING,          -- degenerate dimension
  event_date    DATE,
  line_amount   DECIMAL(12,2),   -- measured at the event, not the line
  event_type    STRING
)
-- grain: one row per (order_id, line_num, event_type)
-- natural key = order_id + line_num + event_type
"""
print(fact_ddl)

# (3) THE GRAIN TEST — this belongs in your CI/data-quality suite:
natural_key = ["order_id", "line_num", "event_type"]
dupes = (order_line_events.groupBy(natural_key).count().filter("count > 1"))
assert dupes.count() == 0, f"GRAIN VIOLATION: {dupes.collect()}"
assert order_line_events.select(natural_key).distinct().count() == order_line_events.count()

# (4) Concrete corruption example: if a duplicate (PAYMENT) row for ORD-5001 line 1
# slips in and nobody tests grain, SUM(line_amount) over 'PAYMENT' doubles revenue
# for that order — silently. Every dashboard, every finance recon, every ML feature
# derived from this table inherits the error. Grain is the FIRST test you write.

# Interview points:
# - Declare grain out loud in one sentence BEFORE any columns: 'one row per order
#   line event'. Every design decision (keys, facts, dimensions, aggregation rules)
#   follows from it. Wrong grain = wrong table, full stop.
# - Degenerate dimensions: the order/invoice NUMBER has descriptive value (joins,
#   drill-to-source, recon) but no attributes of its own — a dimension table would
#   have one row per key and nothing else. Keep the number in the fact; that IS its
#   home. Distinguished from natural keys: they PARTIALLY make up the natural key.
# - Event-grain facts accumulate one row per state transition; an accumulating-snapshot
#   alternative exists for pipeline views (day 14) — tradeoff: volume vs update cost.
print("Q12 OK — grain holds: 5 rows, 5 distinct natural keys")

## Q13 — Data Modeling: Data Mesh & Domain Data Products [ADVANCED]

**Interviewer (Zalando):** A monolithic central data platform can't keep up with 40 product teams. Evaluate **data mesh** as the target architecture.

Cover:
1. The four principles: **domain ownership**, **data as a product**, **self-serve data platform**, **federated computational governance**.
2. What does a "data product" concretely look like (SLOs, contracts, discoverability, an accountable owner)?
3. Map the principles onto Databricks: Unity Catalog catalogs/schemas per domain, lineage, tags, quality rules, Genie spaces.
4. The honest critique: where mesh fails (duplicated storage, cross-domain joins, governance gaps, small-org overhead) — and what companies actually adopt in practice.
5. Your verdict for a 40-team e-commerce org vs a 5-team startup.

In [0]:
# Solution Q13 — data mesh talk track + Databricks mapping

principles = {
    "1. Domain ownership":     "Data is owned by the team that knows it (the orders team owns orders data), end-to-end: ingestion, quality, SLAs, access grants. The central team stops being a data help desk and becomes a platform team.",
    "2. Data as a product":    "Each dataset is a PRODUCT with a named owner, versioned contract, SLOs (freshness, completeness), discoverable metadata, and a support channel. Consumers are customers, not ticket-filers.",
    "3. Self-serve platform":  "Domains create data products WITHOUT filing tickets: automated infra (pipelines, catalogs, monitoring), golden paths, paved-road templates. Infrastructure as a commodity.",
    "4. Federated governance": "Global policies (PII, naming, retention, quality standards) computed and enforced AUTOMATICALLY at the platform layer (UC tags, ABAC policies) — governance without a central bottleneck committee.",
}
for k, v in principles.items():
    print(f"{k}\n  {v.strip()}\n")

DATABRICKS_MAP = """
MAPPING TO DATABRICKS (say this in the interview):
- Domain ownership   -> one UC catalog (or schema) per domain: orders_gold, payments_gold.
                        Domain teams have CREATE/MANAGE on their catalog only.
- Data as a product  -> UC-level data contracts (schema + constraints), tags,
                        lineage, column comments as documentation, model (Genie) spaces
                        as the product's 'API' for business questions.
- Self-serve         -> Lakeflow pipelines + jobs as templates, Auto Loader golden
                        path, workspace-scoped CI/CD per domain, staging catalogs.
- Federated govern.  -> UC ABAC policies + governed tags applied globally, row/column
                        access policies, audit via system tables — enforced once,
                        inherited everywhere.

THE HONEST CRITIQUE (interviewers reward this):
- Cross-domain joins become cross-team NEGOTIATIONS: 'join my product' = contract.
  Shared dimensions must be central (conformed) or you re-create the monolith.
- Duplication: every domain copies reference data; storage is cheap, CONSISTENCY is not.
- Small orgs: 5 teams do NOT need mesh — the coordination overhead exceeds the
  bottleneck it removes. Mesh is an org-scaling tool, not a tech toy.
- Practice: most adopters land on a PRAGMATIC mesh — central platform team + strong
  domain ownership + a few conformed shared products (customers, revenue).

VERDICT: 40-team org -> pragmatic mesh (phased: pick 3 pilot domains first).
5-team startup -> one central lakehouse + medallion; revisit mesh past ~15-20 teams.
"""
print(DATABRICKS_MAP)
print("Q13 OK — four principles + platform mapping + critique + verdict")

## Q14 — Snowflake: Iceberg Tables & Hybrid Tables (2025-26 Trend) [ADVANCED]

**Interviewer:** Snowflake now supports Apache Iceberg natively. Explain the Iceberg table flavors in Snowflake and when to use each — plus what hybrid tables are for.

Cover:
1. **Snowflake-managed Iceberg tables** vs **external Iceberg tables** (BYOC — bring your own catalog): who owns the files, who owns the metadata, and what is the **catalog integration** (AWS Glue, object-store catalog, Polaris/REST)?
2. Why would a company WANT Iceberg in Snowflake (no lock-in, other engines reading the same files, exit strategy) — and what it gives up (some Snowflake-only features).
3. **Hybrid tables** (HTAP): row-based storage, primary keys, indexes — what workload actually needs them vs a standard columnar table?
4. Gotchas of each flavor: feature parity (cloning, Time Travel semantics, streaming), maintenance burden, and cost implications.

In [0]:
# Solution Q14 — talk track + SQL sketches (Snowflake dialect, for reference)

SQL_SKETCH = """
-- Snowflake-managed Iceberg: Snowflake owns catalog+metadata, files in YOUR cloud storage
CREATE ICEBERG TABLE analytics.events (
  event_ts TIMESTAMP_NTZ, user_id NUMBER, kind STRING
)
  CATALOG = SNOWFLAKE
  EXTERNAL_VOLUME = my_s3_vol
  BASE_LOCATION = 'analytics/events/';

-- External Iceberg (BYOC): an EXISTING Iceberg catalog/table (Glue/Polaris/REST)
-- Snowflake reads (and can write, 2025 GA) without owning the metadata.
CREATE ICEBERG TABLE sales.ice_orders
  CATALOG_REFERENCED_DATABASE = GLUE_CAT
  EXTERNAL_VOLUME = my_s3_vol
  CATALOG = MY_GLUE_INTEGRATION
  CATALOG_NAMESPACE = 'sales' CATALOG_TABLE_NAME = 'orders';

-- Hybrid table (HTAP): row store + PK + secondary indexes
CREATE HYBRID TABLE billing.wallets (
  wallet_id NUMBER PRIMARY KEY AUTOINCREMENT,
  user_id NUMBER, balance NUMBER(12,2),
  INDEX idx_user (user_id)
);
"""
print(SQL_SKETCH)

TALK_TRACK = """
1) TWO ICEBERG FLAVORS — the ownership split is the exam answer:
   - MANAGED: you own the FILES (your S3 volume), Snowflake owns the CATALOG +
     metadata. You get near-full Snowflake UX (cloning-ish, streaming loads) while
     the files stay in open Parquet+Iceberg format — the exit door is 'another engine
     can read my files if I register the catalog'.
   - EXTERNAL (BYOC): YOU own the catalog too (Glue/Polaris/REST/Iceberg REST spec).
     Snowflake is just one engine among Spark/Trino/Flink reading the same table.
     Maximum openness, but Snowflake-only features (Time Travel beyond snapshots,
     some DML optimizations) depend on what your catalog allows.
2) WHY ADOPT: procurement/exit strategy (files aren't captive), multi-engine reads
   (one copy, many engines), regulatory demands for open formats. What you give up:
   certain Snowflake magic (zero-copy cloning semantics, near-instant RESTORE) is
   bounded by Iceberg snapshot mechanics instead.
3) HYBRID TABLES = the OLTP escape hatch inside Snowflake: row-oriented pages,
     enforceable primary keys, secondary indexes, point lookups + updates at
     millisecond latency. Use for stateful apps (wallets, sessions, rate-limit
     counters). NOT for analytics — columnar tables crush hybrid for scans. Don't
     build your lakehouse on them.
4) GOTCHAS: managed = simplest ops; external = you now run snapshot expiry,
     compaction, and catalog hygiene YOURSELF (same ops burden as on-prem Iceberg).
     Cost: Iceberg writes may skip Snowflake micro-partition optimizations; check
     replication/failover support per flavor for DR (see Q15).
"""
print(TALK_TRACK)
print("Q14 OK — flavors, ownership split, HTAP, gotchas")

## Q15 — Snowflake: Replication & Failover for Business Continuity [ADVANCED]

**Interviewer (Goldman Sachs, via Snowflake):** Design cross-region disaster recovery for a Snowflake account serving a regulated business.

Cover:
1. **Replication groups** vs **failover groups** — what exactly does `FAILOVER` add over plain replication, and what does a **switchover** do?
2. Write the DDL: replicate databases + shares to a secondary region on a schedule.
3. What replicates (storage, Time Travel history, shares) and what does NOT (warehouses, resource monitors, pipes/tasks in the same form, users/roles/grants — depending on account replication config)?
4. The DR drill: walk through promoting the secondary. What breaks first, and how do you practice without a real outage?
5. Cost: how is replicated storage and replication activity billed?

In [0]:
# Solution Q15 — DDL + DR walk-through (Snowflake dialect, for reference)

DDL = """
-- (primary account, once)
CREATE FAILOVER GROUP fg_prod
  FOR DATABASES analytics, finance
      SHARES share_sales_eu, share_partners
  ALLOW_ACCESS_INTEGRATIONS = ALL
  REPLICATION_SCHEDULE = '10 MINUTE';

-- (secondary account)
CREATE FAILOVER GROUP fg_prod AS REPLICA OF org.primary_account.fg_prod;

-- DR drill / real failover (IN THE SECONDARY):
ALTER FAILOVER GROUP fg_prod PRIMARY;     -- switchover: becomes writable primary
-- then fail writes on the old primary back or run it read-only (split-brain guard).
"""
print(DDL)

TALK_TRACK = """
1) REPLICATION GROUP = asynchronous copy of databases/shares to secondary accounts
   on a schedule; the SECONDARY is READ-ONLY. FAILOVER GROUP = replication group PLUS
   the right to promote (switchover). If you need only a read replica in another
   region (latency), replication group is enough; if you need DR, failover group.
2) SWITCHOVER is bidirectional: the roles of primary/secondary flip, and after the
   old primary comes back it can REVERSE the failover — that's why failover groups
   beat 'replicate + manual CREATE DATABASE FROM SHARE' scripts.
3) WHAT REPLICATES: data + (configurably) grants, shares, policies, tags, network
   rules, and Time Travel history (up to the TT retention, streamed). WHAT DOESN'T,
   by default: warehouses (compute), resource monitors, pipes and tasks have their
   own replication/refresh rules, and users/roles only with SCIM-level account
   replication. DR runbook must RE-CREATE compute and re-enable pipes/tasks after
   switchover — that's the thing people forget in the drill.
4) DRILL: schedule a quarterly game day — in a NON-PROD replica group, run
   ALTER ... PRIMARY, repoint one service, verify, then reverse. Measure RPO:
   replication_schedule = your RPO ceiling (10 min schedule = up to 10 min data
   loss). RTO = DNS/repoint + warehouse provisioning + cache warmup.
5) COST: replicated storage is billed in the SECONDARY region; each replication
   job consumes DATA TRANSFER out of the primary region (egress) + serverless-like
   replication compute. Frequency x database size drives the bill — replicate the
   3 databases that matter, not all 30.
"""
print(TALK_TRACK)
print("Q15 OK — failover group DDL + RPO/RTO + billing")

## Q16 — Monitoring & Orchestration: Airflow vs Dagster vs Prefect [ADVANCED]

**Interviewer (startup CTO):** We're picking a modern orchestrator. Compare the three and recommend.

Cover:
1. Core mental model of each: **DAG-of-tasks** (Airflow) vs **asset/graph-based** (Dagster) vs **dynamic pythonic flows** (Prefect) — and what falls out of each model.
2. Where each shines: Airflow maturity/ecosystem/managed offering + Databricks integrations; Dagster asset lineage, freshness policies, native dbt/SQLMesh; Prefect developer experience, runtime-dynamic DAGs, easy local dev.
3. The Databricks-native question: when do you not need a third-party orchestrator at all (Databricks Jobs: task types, Run If dependencies, file-arrival/table-update triggers) — and when you still do (cross-system, non-Spark tasks, org-wide scheduling).
4. Concrete recommendations for: (a) a Spark-only shop on Databricks, (b) a multi-tool org with dbt + Snowflake + ML training.

In [0]:
# Solution Q16 — comparison matrix + recommendations (talk track)

matrix = [
    ("Core model",       "DAG of tasks (operators)",        "asset graph (software-defined assets)",      "python functions + runtime flows"),
    ("Unit of thought",  "the TASK (what to run)",           "the ASSET (what to produce)",                "the FLOW (pythonic steps)"),
    ("Lineage",          "task-level, needs dbt/asset sync",  "first-class: asset lineage + column lineage", "flow/task level via UI"),
    ("Dynamic DAGs",      "hard (deferred, mapped operators)", "supported (partitions/backfills)",            "trivial — DAG exists only at runtime"),
    ("Freshness/SLA",    "SLAs on tasks, alerts on misses",   "freshness POLICIES on assets",                "automations + SLA blocks"),
    ("dbt integration",   "mature (Cosmos/Astronomer)",        "native asset materialization",                "via dbt CLI blocks"),
    ("Databricks",       "deep, battle-tested operators",     "solid Databricks asset IO",                   "standard"),
    ("Local dev / DX",    "needs a running scheduler",        "best-in-class (dg CLI, dagster dev)",         "excellent (decorator-first)"),
    ("Managed option",   "MWAA / Astronomer",               "Dagster+ (Cloud)",                             "Prefect Cloud"),
]
widths = [18, 30, 34, 34]
print(f"{'Aspect':<18} {'Airflow':<30} {'Dagster':<34} {'Prefect':<34}")
print("-" * 116)
for row in matrix:
    print(" | ".join(str(v)[:w].ljust(w) for v, w in zip(row, widths)))

RECS = """
RECOMMENDATIONS (deliver these crisply):
(a) Spark-only shop on Databricks: DO NOT bolt on an orchestrator yet — Databricks
    Jobs gives task types (notebook, SQL, pipeline, python), Run If dependencies
    (ALL_SUCCESS / ON_FAILURE / ALL_FAILED), retries with backoff, and triggers
    (cron, file arrival, table update via Lakeflow). Orchestration lives where the
    data lives. Add one only when cross-system glue appears.
(b) Multi-tool org (dbt + Snowflake + ML training + external APIs): Dagster if your
    mental model is ASSETS and you want lineage + freshness policies as the product
    (analytics-engineering-first org); Airflow if ops maturity, hiring pool, and a
    huge operator ecosystem matter most (classic data-eng org); Prefect when the
    team is python-native, DAGs change at runtime, and DX is king (ML/platform org).

The 2025-26 trend to name-drop: asset-oriented orchestration converging with
    data platforms (Dagster assets / Databricks Tables+Lakeflow datasets),
    and 'declarative' orchestration (declarations of what should exist) vs
    imperative task lists.
"""
print(RECS)
print("Q16 OK — matrix + two-scenario recommendation")

## Q17 — Monitoring & Debugging: Delta Time-Travel Forensics [ADVANCED]

**Interviewer (the 3 AM pager):** A dashboard shows garbage since 03:14. A Delta table was appended with bad rows overnight. **Find the write, fix it, prove it, and prevent it.**

1. `DESCRIBE HISTORY` — which columns matter (version, operation, timestamp, user, `operationParameters`) and how to pinpoint the exact bad commit?
2. Repair options: `RESTORE ... TO VERSION AS OF` vs a MERGE-based delete vs a filtered view — tradeoffs (simplicity vs history preservation vs new commit only).
3. `VACUUM`: why the bad data is not really gone after the fix, retention windows, and why vacuuming before verifying the restore is dangerous.
4. Prevention: what guard stops the 3 AM append from happening again (contract checks, NOT NULL/CHECK constraints, write permissions, quarantine pattern)?

The demo below creates a table, appends a poisoned batch, and restores — fully runnable.

In [0]:
# Solution Q17 — live forensics: history -> restore -> verify (path-based Delta table)
from pyspark.sql import functions as F

path = "/Volumes/workspace/default/etl_test_volume/d18_q17/events"
dbutils.fs.rm(path, True)  # idempotent re-run

# v0: the good write
good = spark.createDataFrame([(1, "login"), (2, "login"), (3, "purchase")], ["user_id", "event"])
good.write.format("delta").mode("overwrite").save(path)
# v1: the 3 AM poisoned append (upstream mis-mapped event labels)
bad = spark.createDataFrame([(4, "loggin"), (5, "loign")], ["user_id", "event"])
bad.write.format("delta").mode("append").save(path)

# 1) FIND THE WRITE — newest first, so ORDER/scan for the suspicious window
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "operation", "user").show(truncate=False)
# In the interview: filter on timestamp BETWEEN 03:10 AND 03:20, look at operation,
# operationParameters (mode -> Append), user, and cluster/job context in the job system tables.

# 2) REPAIR — RESTORE is the surgical, zero-code option: it is ITSELF a commit,
# so you get a clean audit trail AND the old versions remain queryable.
spark.sql(f"RESTORE TABLE delta.`{path}` TO VERSION AS OF 0")

# 3) PROVE IT
rows = spark.sql(f"SELECT user_id, event FROM delta.`{path}` ORDER BY user_id").collect()
assert [(r.user_id, r.event) for r in rows] == [(1, "login"), (2, "login"), (3, "purchase")]
print("restored rows:", [(r.user_id, r.event) for r in rows])

# VACUUM talk track:
# - RESTORE doesn't delete the bad files; they stay until VACUUM + retention window
#   (default 7 days) passes — which is GOOD: it's your safety net, and required for
#   auditing a regulated incident. Don't vacuum the day of the incident.
# - VACUUM BEFORE verifying the restore (or with 0 retention) can orphan you from
#   every fallback version — verify queries + downstream dashboards FIRST, always.
# Tradeoffs to state: RESTORE (simple, new commit, keeps history) vs MERGE-delete
#   (surgical rows, more code, still a commit) vs filtered VIEW (fastest, but
#   buries the problem — the table stays dirty; last resort).
# 4) PREVENTION: schema + data contracts on write (expect-event checks),
#    CHECK constraints in DDL, least-privilege write permissions (the 3 AM job
#    should write via its own principal, auditable), and a quarantine first hop
#    (Q5) so bad batches never touch the serving table.
dbutils.fs.rm(path, True)  # clean up demo artifacts
print("Q17 OK — bad append found, restored to v0, verified, prevented")

## Q18 — Monitoring & Governance: Semantic Layer / Metric Store [HARD]

**Interviewer (GenAI-era analytics platform):** "Revenue" is computed six different ways in six dashboards. Design a **metric layer** so there is exactly one definition of every business metric.

Cover:
1. Why metric drift happens, and the cost of ad-hoc metric SQL (conflicting dashboards, slow audits, lost trust).
2. Databricks-native options: **Unity Catalog metric views** (`CREATE METRIC`, DBSQL) and **AI/BI Genie** — how they centralize + govern definitions.
3. How **dbt metrics / semantic layer** and **Headless BI** (Cube etc.) compare — and where does the metric physically execute?
4. Implement a governed **metric-registry pattern** runnable on Spark (below), and explain how the registry becomes the enforcement point (only the registry defines metrics; dashboards consume by NAME).
5. Why do LLM/GenAI analytics make semantic layers MORE important (grounding text-to-SQL, one canonical answer per question)?

In [0]:
# Solution Q18 — governed metric registry (runnable pattern) + platform talk track
from pyspark.sql import functions as F

# The source of truth data (kept as temp views here; a governed UC table in production)
orders = spark.createDataFrame([
    ("2026-10-28", "PRO", 3, 300.0), ("2026-10-28", "FREE", 10, 90.0),
    ("2026-10-29", "PRO", 4, 410.0), ("2026-10-29", "FREE", 12, 120.0),
    ("2026-10-30", "PRO", 5, 520.0), ("2026-10-30", "FREE", 9, 85.0),
], ["order_date", "plan", "order_count", "revenue"])
orders.createOrReplaceTempView("orders_src")

# THE REGISTRY: metrics defined ONCE — named, owned, unit-tagged, versioned.
# Dashboards NEVER write SUM(revenue) again; they call get_metric('total_revenue').
registry = [
    ("total_revenue",   "orders_src", "SUM(revenue)",                 "finance", "USD"),
    ("total_orders",    "orders_src", "SUM(order_count)",             "growth",  "count"),
    ("avg_order_value", "orders_src", "AVG(revenue / order_count)",   "growth",  "USD"),
]

def get_metric(metric_name, group_cols="plan"):
    """The ONLY way a consumer gets a number: ask by metric NAME."""
    m = next(r for r in registry if r[0] == metric_name)
    return spark.sql(f"""
        SELECT {group_cols}, {m[2]} AS {m[0]}
        FROM {m[1]}
        GROUP BY {group_cols} ORDER BY {group_cols}
    """)

rev = {r["plan"]: r["total_revenue"] for r in get_metric("total_revenue").collect()}
cnt = {r["plan"]: r["total_orders"] for r in get_metric("total_orders").collect()}
print("total_revenue by plan:", rev)
print("total_orders by plan:  ", cnt)
assert rev == {"FREE": 295.0, "PRO": 1230.0}
assert cnt == {"FREE": 31, "PRO": 12}

TALK_TRACK = """
1) METRIC DRIFT: every dashboard team re-derives 'revenue' (gross? net? refunds?
   which currency date?), answers diverge, and audits take weeks re-proving numbers.
   The fix is ORGANIZATIONAL as much as technical: ONE definition, ONE owner,
   versioned changes.
2) DATABRICKS-NATIVE: Unity Catalog METRIC VIEWS — CREATE METRIC ... ON TABLE ...
   AS SUM(revenue) — define it once, query by name (my_metric(dim)) in DBSQL,
   governed + inherited through UC lineage; AI/BI Genie spaces then expose governed
   data conversationally. (Metric views are a DBSQL feature — the Spark registry
   pattern above is the same idea, runnable on any compute.)
3) ECOSYSTEM: dbt metrics/semantic layer defines metrics with the dbt project;
   Headless-BI (Cube) serves metrics via an API and pushes down to the warehouse
     -> metric EXECUTES where the data lives; the layer only translates.
   Key interview line: a semantic layer that copies data is a DATA MART with extra
   steps — the definition moves, not the bytes.
4) ENFORCEMENT: registry = the only write-path for metric SQL, CI-tested (does the
   metric still return expected values?), changes reviewed by the metric OWNER,
   dashboards/Genie consume by NAME ONLY. Metric version bump = semver + change log.
5) GENAI: text-to-SQL/agents hallucinate worse on ambiguous metrics; a governed
   metric store is the GROUNDING layer — the LLM picks from registered metrics,
   so 'revenue' has one answer. Semantic layers went from BI nicety to GenAI
   prerequisite in two years.
"""
print(TALK_TRACK)
print("Q18 OK — registry-driven metrics + platform mapping + GenAI angle")

## 📊 Day 18 Score Card

| Q | Category | Topic | Difficulty | Solved? (✓/✗/~) | Notes |
|---|----------|-------|------------|------------------|-------|
| 1 | SQL Advanced | Market-basket pairs + lift | ADVANCED |  |  |
| 2 | SQL Advanced | GROUPING SETS subtotals | ADVANCED |  |  |
| 3 | Python | Median from data stream (two heaps) | ADVANCED |  |  |
| 4 | Python | Course schedule (topological sort) | ADVANCED |  |  |
| 5 | PySpark | Corrupt-record quarantine (PERMISSIVE) | ADVANCED |  |  |
| 6 | PySpark | Robust anomaly detection (median + MAD) | ADVANCED |  |  |
| 7 | Spark Opt | Dynamic resource allocation + executor sizing | ADVANCED |  |  |
| 8 | Spark Opt | File stats, data skipping, compression | ADVANCED |  |  |
| 9 | ETL | Backpressure in streaming pipelines | ADVANCED |  |  |
| 10 | ETL | Reverse ETL hash-diff CDC extract | ADVANCED |  |  |
| 11 | ETL | Delta vs Iceberg vs Hudi | HARD |  |  |
| 12 | Data Modeling | Grain declaration + degenerate dimensions | ADVANCED |  |  |
| 13 | Data Modeling | Data mesh & data products | ADVANCED |  |  |
| 14 | Snowflake | Iceberg tables & hybrid tables | ADVANCED |  |  |
| 15 | Snowflake | Replication & failover groups | ADVANCED |  |  |
| 16 | Monitoring | Airflow vs Dagster vs Prefect | ADVANCED |  |  |
| 17 | Monitoring | Delta time-travel forensics | ADVANCED |  |  |
| 18 | Monitoring | Semantic layer / metric store | HARD |  |  |

**Scoring:** ✓ solved cleanly · ~ solved with hints · ✗ struggled.

**Day 18 recap:** 16 ADVANCED + 2 HARD — Week 3 Day 4. Review every ✗ tomorrow before Day 19.